## 1. Setup & Imports

In [ ]:
# Core imports
import sys
import os
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Visualization
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, Circle
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Interactive widgets
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

# RL components
from stable_baselines3 import PPO
from environments.simple_trading_env import SimpleTradingEnv

print('✓ Imports successful')

## 2. Configuration

In [ ]:
# === CONFIGURATION ===
DATA_SYMBOL = 'BTCUSDT'
DATA_TIMEFRAME = '5m'
DATA_PATH = f'data/binance-{DATA_SYMBOL}-{DATA_TIMEFRAME}.pkl'
LOOKBACK_WINDOW = 288  # Bars to show in chart (24 hours for 5m)

# Visualization settings
CONFIG = {
    'chart_height': 800,
    'chart_width': 1400,
    'feature_matrix_width': 14,
    'feature_matrix_height': 8,
    'play_speed_default': 1.0,
    'color_scheme': {
        'bullish': '#00C853',
        'neutral': '#FFD600',
        'bearish': '#D50000',
        'inactive': '#BDBDBD',
    },
    'feature_thresholds': {
        'bullish': 0.3,
        'bearish': -0.3,
    }
}

print(f'✓ Configuration loaded')
print(f'  Data: {DATA_SYMBOL} {DATA_TIMEFRAME}')
print(f'  Lookback: {LOOKBACK_WINDOW} bars')

## 3. Load Data & Create Environment

In [ ]:
# Load data
df = pd.read_pickle(DATA_PATH)
print(f'✓ Loaded {len(df):,} rows')

# Select subset for visualization (avoid loading full dataset)
start_idx = 0  # Start from beginning or specific index
end_idx = min(10_000, len(df))  # First 10k rows for quick testing
viz_data = df.iloc[start_idx:end_idx].copy()
print(f'✓ Using rows {start_idx:,} to {end_idx:,} for visualization')

# Create environment
env = SimpleTradingEnv(
    data=viz_data,
    lookback_window=LOOKBACK_WINDOW,
    initial_balance=10_000,
)

# Reset environment
obs = env.reset()
print(f'✓ Environment created and reset')
print(f'  Initial step: {env.current_step}')
print(f'  Max steps: {len(env.data) - LOOKBACK_WINDOW}')

## 5. Interactive Visualizer Class

In [ ]:
class InteractiveEnvVisualizer:
    """Interactive visualizer for stepping through trading environment"""
    
    def __init__(self, env, model=None, config=None):
        self.env = env
        self.model = model
        self.config = config or CONFIG
        
        # State tracking
        self.last_action = None
        self.last_reward = 0.0
        self.last_info = {}
        self.max_step_reached = self.env.current_step  # Track furthest step reached
        
        # Widgets (created in display())
        self.widgets = {}
        
    def step_forward(self, action=None):
        """Advance one step"""
        # Get action from model or manual override
        if action is None and self.model is not None:
            obs = self.env._get_obs()
            action_pred, _ = self.model.predict(obs, deterministic=True)
            action = int(action_pred[0])
        elif action is None:
            action = 0  # Default to HOLD
        
        step_before = self.env.current_step
        print(f'DEBUG: Before step - current_step={step_before}, action={action}')
        
        obs, reward, done, truncated, info = self.env.step([action])
        
        step_after = self.env.current_step
        print(f'DEBUG: After step - current_step={step_after}, advanced={step_after > step_before}')
        
        if step_after > step_before:
            self.last_action = action
            self.last_reward = reward
            self.last_info = info
            if step_after > self.max_step_reached:
                self.max_step_reached = step_after
        else:
            print(f'⚠️ Invalid action {action} - step did not advance')
        
        if done or truncated:
            print('⚠️  Episode ended')
            return False
        
        return True
    
    def jump_to_step(self, target_step):
        """Jump backward by modifying env.current_step"""
        # Can only go backward, not forward
        if target_step > self.env.current_step:
            print(f'⚠️ Cannot jump forward - use Next button')
            return False
        
        # Cannot go below lookback window
        min_step = self.env.lookback_window
        if target_step < min_step:
            print(f'⚠️ Cannot go below lookback window ({min_step})')
            target_step = min_step
        
        # Directly set the step
        self.env.current_step = target_step
        return True
        
    def plot_price_chart(self):
        """Plot candlestick chart with Volume Profile"""
        current_step = self.env.current_step
        
        # Get window of data to display
        window_start = max(0, current_step - self.env.lookback_window)
        window_end = current_step + 1
        window_data = self.env.data.iloc[window_start:window_end]
        
        # Create subplots
        fig = make_subplots(
            rows=4, cols=1,
            row_heights=[0.6, 0.15, 0.15, 0.1],
            subplot_titles=('Price & Volume Profile', 'Position Size', 'Equity', 'Reward'),
            vertical_spacing=0.05,
            shared_xaxes=True
        )
        
        # 1. Candlestick chart
        candlestick = go.Candlestick(
            x=window_data.index,
            open=window_data['open'],
            high=window_data['high'],
            low=window_data['low'],
            close=window_data['close'],
            name='Price'
        )
        fig.add_trace(candlestick, row=1, col=1)
        
        # Highlight current bar with invisible marker for padding
        current_bar = window_data.iloc[-1]
        fig.add_trace(
            go.Scatter(
                x=[current_bar.name],
                y=[current_bar['close']],
                mode='markers',
                marker=dict(size=15, color='yellow', symbol='diamond', opacity=0),
                name='Current',
                showlegend=False
            ),
            row=1, col=1
        )
        
        # Add SL/TP lines if in position
        if self.env.broker.position_size != 0:
            current_price = current_bar['close']
            
            # Stop Loss line
            if self.env.broker.stop_loss_price is not None:
                fig.add_hline(
                    y=self.env.broker.stop_loss_price,
                    line_dash="dash",
                    line_color="red",
                    annotation_text="SL",
                    annotation_position="right",
                    row=1, col=1
                )
            
            # Take Profit line
            if self.env.broker.take_profit_price is not None:
                fig.add_hline(
                    y=self.env.broker.take_profit_price,
                    line_dash="dash",
                    line_color="green",
                    annotation_text="TP",
                    annotation_position="right",
                    row=1, col=1
                )
            
            # Entry price line
            if self.env.broker.avg_entry_price is not None:
                fig.add_hline(
                    y=self.env.broker.avg_entry_price,
                    line_dash="dot",
                    line_color="blue",
                    annotation_text="Entry",
                    annotation_position="right",
                    row=1, col=1
                )
        
        # 2. Position size over time (from env.history)
        if len(self.env.history) > 0:
            history_steps = [h['step'] for h in self.env.history]
            history_positions = [h['position_size'] for h in self.env.history]
            fig.add_trace(
                go.Scatter(
                    x=history_steps,
                    y=history_positions,
                    mode='lines',
                    name='Position',
                    line=dict(color='blue', width=2),
                    showlegend=False
                ),
                row=2, col=1
            )
        
        # 3. Equity curve (from env.history)
        if len(self.env.history) > 0:
            history_steps = [h['step'] for h in self.env.history]
            history_equity = [h['equity'] for h in self.env.history]
            fig.add_trace(
                go.Scatter(
                    x=history_steps,
                    y=history_equity,
                    mode='lines',
                    name='Equity',
                    line=dict(color='green', width=2),
                    showlegend=False
                ),
                row=3, col=1
            )
        
        # 4. Reward signal (from env.history)
        if len(self.env.history) > 0:
            history_steps = [h['step'] for h in self.env.history]
            history_rewards = [h['reward'] for h in self.env.history]
            reward_colors = ['green' if r > 0 else 'red' if r < 0 else 'gray' for r in history_rewards]
            fig.add_trace(
                go.Bar(
                    x=history_steps,
                    y=history_rewards,
                    name='Reward',
                    marker_color=reward_colors,
                    showlegend=False
                ),
                row=4, col=1
            )
        
        # Update layout
        fig.update_layout(
            height=self.config['chart_height'],
            width=self.config['chart_width'],
            showlegend=False,
            title_text=f'Step {current_step:,} / {len(self.env.data) - self.env.lookback_window:,}',
            hovermode='x unified'
        )
        
        fig.update_xaxes(title_text='Time', row=4, col=1)
        fig.update_yaxes(title_text='Price', row=1, col=1)
        fig.update_yaxes(title_text='Position', row=2, col=1)
        fig.update_yaxes(title_text='Equity ($)', row=3, col=1)
        fig.update_yaxes(title_text='Reward', row=4, col=1)
        
        return fig
    
    def plot_feature_matrix(self):
        """Plot feature activation heatmap"""
        obs = self.env._get_obs()
        
        # Extract last timestep from each feature group
        features = {
            'Micro Temporal': obs['micro_temporal'][-1],  # Last timestep
            'Micro Spatial': obs['micro_spatial'][-1],
            'Meso': obs['meso_patterns'][-1],
            'Macro': obs['macro_patterns'][-1],
            'Account': obs['account_state'],
            'Position': obs['position_info'],
        }
        
        # Create figure
        fig, ax = plt.subplots(
            figsize=(self.config['feature_matrix_width'], self.config['feature_matrix_height'])
        )
        
        # Plot each feature group as row of colored circles
        y_pos = 0
        for group_name, values in features.items():
            values_flat = values.flatten()
            
            for i, val in enumerate(values_flat[:20]):  # Show first 20 features max
                # Determine color based on value
                if val > self.config['feature_thresholds']['bullish']:
                    color = self.config['color_scheme']['bullish']
                elif val < self.config['feature_thresholds']['bearish']:
                    color = self.config['color_scheme']['bearish']
                elif abs(val) > 0.01:
                    color = self.config['color_scheme']['neutral']
                else:
                    color = self.config['color_scheme']['inactive']
                
                # Draw circle
                circle = Circle((i, y_pos), 0.3, color=color, ec='black', linewidth=1)
                ax.add_patch(circle)
            
            # Add label
            ax.text(-1, y_pos, group_name, ha='right', va='center', fontsize=10, fontweight='bold')
            
            y_pos += 1
        
        # Set limits and remove axes
        ax.set_xlim(-2, 20)
        ax.set_ylim(-0.5, y_pos - 0.5)
        ax.set_aspect('equal')
        ax.axis('off')
        
        ax.set_title('Feature Activation Matrix', fontsize=14, fontweight='bold', pad=20)
        
        plt.tight_layout()
        return fig
    
    def get_state_info_html(self):
        """Get formatted state information as HTML"""
        current_step = self.env.current_step
        balance = self.env.broker.current_balance
        equity = self.env.broker.equity
        position_size = self.env.broker.position_size
        
        # Get last action and reward
        action = self.last_action
        reward = self.last_reward
        
        action_names = ['HOLD', 'LONG', 'SHORT']
        action_str = action_names[action] if action is not None else 'None'
        
        position_str = f"{position_size:.3f} BTC" if position_size != 0 else "FLAT"
        
        balance_change = ((balance / 10000) - 1) * 100
        
        # Show max step from data
        max_step = len(self.env.data) - self.env.lookback_window - 1
        
        # Show SL/TP info if in position
        sl_tp_info = ""
        if position_size != 0:
            if self.env.broker.stop_loss_price is not None:
                sl_distance = ((self.env.broker.stop_loss_price / self.env.data.iloc[current_step]['close']) - 1) * 100
                sl_tp_info += f"<b>Stop Loss:</b> ${self.env.broker.stop_loss_price:.2f} ({sl_distance:+.2f}%)<br>"
            if self.env.broker.take_profit_price is not None:
                tp_distance = ((self.env.broker.take_profit_price / self.env.data.iloc[current_step]['close']) - 1) * 100
                sl_tp_info += f"<b>Take Profit:</b> ${self.env.broker.take_profit_price:.2f} ({tp_distance:+.2f}%)<br>"
        
        html = f"""
        <div style="font-family: monospace; font-size: 13px; line-height: 1.6;">
            <b>Step:</b> {current_step:,} / {max_step:,}<br>
            <b>Action:</b> <span style="color: {'blue' if action == 1 else 'red' if action == 2 else 'gray'};">{action_str}</span><br>
            <b>Reward:</b> <span style="color: {'green' if reward > 0 else 'red' if reward < 0 else 'gray'};">{reward:+.4f}</span><br>
            <b>Position:</b> {position_str}<br>
            {sl_tp_info}
            <b>Balance:</b> ${balance:,.2f} (<span style="color: {'green' if balance_change >= 0 else 'red'};">{balance_change:+.2f}%</span>)<br>
            <b>Equity:</b> ${equity:,.2f}<br>
        </div>
        """
        
        return html
    
    def update_display(self):
        """Update all visualization components"""
        slider = self.widgets['slider']
        # Remove observer to avoid triggering on_slider_change
        slider.unobserve(self.on_slider_change, names='value')
        with self.widgets['output']:
            clear_output(wait=True)
            # Update slider to reflect current position
            slider.value = self.env.current_step
            slider.max = self.max_step_reached
            # Show state info
            display(HTML(self.get_state_info_html()))
            # Show price chart
            fig = self.plot_price_chart()
            fig.show()
            # Show feature matrix
            fig_features = self.plot_feature_matrix()
            plt.show()
            plt.close(fig_features)
        # Re-add observer
        slider.observe(self.on_slider_change, names='value')
    
    def on_next_click(self, btn):
        print('Clicked Next button')
        """Handle next button click"""
        action = self.widgets['action_dropdown'].value
        if action == 'Agent':
            action = None
        else:
            action = ['HOLD', 'LONG', 'SHORT'].index(action)
        
        success = self.step_forward(action=action)
        if success:
            self.update_display()
    
    def on_slider_change(self, change):
        """Handle slider change - only allows going backward"""
        target_step = change['new']
        current_step = self.env.current_step
        
        if target_step < current_step:
            # Going backward - allowed
            if self.jump_to_step(target_step):
                self.update_display()
        elif target_step > current_step:
            # Trying to go forward - not allowed
            print('⚠️ Use Next button to go forward')
            # Reset slider to current position
            self.widgets['slider'].value = current_step
    
    def display(self):
        """Launch interactive dashboard"""
        # Create widgets
        btn_next = widgets.Button(description='Next ▶', button_style='success')
        
        action_options = ['Agent', 'HOLD', 'LONG', 'SHORT'] if self.model else ['HOLD', 'LONG', 'SHORT']
        action_dropdown = widgets.Dropdown(
            options=action_options,
            value=action_options[0],
            description='Action:'
        )
        
        # Slider for backward navigation
        slider = widgets.IntSlider(
            value=self.env.current_step,
            min=self.env.lookback_window,
            max=self.max_step_reached,
            step=1,
            description='Step:',
            continuous_update=False,
            layout=widgets.Layout(width='600px')
        )
        
        output = widgets.Output()
        
        # Store widgets
        self.widgets = {
            'btn_next': btn_next,
            'action_dropdown': action_dropdown,
            'slider': slider,
            'output': output
        }
        
        # Connect callbacks
        btn_next.on_click(self.on_next_click)
        slider.observe(self.on_slider_change, names='value')
        
        # Layout
        controls = widgets.HBox([btn_next, action_dropdown])
        ui = widgets.VBox([controls, slider, output])
        
        # Display
        display(ui)
        self.update_display()

print('✓ InteractiveEnvVisualizer class defined')

## 6. Launch Visualizer

In [ ]:
# Create visualizer instance
viz = InteractiveEnvVisualizer(env, config=CONFIG)

# Launch interactive dashboard
viz.display()